# Tutorial 4 - Downloading and preparing samples from Recount3

This tutorial shows how to download real RNA-seq samples from the `Recount3 <https://rna.recount.bio/>`__ platform and prepare them for use with the bulkdgd model. As a concrete example, we download one TCGA cancer type - glioblastoma multiforme (GBM) - together with three independent SRA studies profiling glioblastoma patient samples (found by searching the SRA metadata for "glioblastoma"/"glioma" and picking studies that represent genuine patient-tumor cohorts). The same steps apply to any single GTEx tissue, TCGA cancer type, or SRA study - just change the ``recount3_project_name``/``recount3_samples_category`` pairs below.

Once downloaded and preprocessed, the resulting samples can be fed directly into :doc:`Tutorial 1 <tutorial_1>` to find their representations in the bulkdgd model's latent space.

In [1]:
# Import from the standard library.
import logging as log

# Import from third-party libraries.
import pandas as pd

# Import from 'bulkdgd'.
from bulkdgd import ioutil, recount3




# Set the logging options so that every message of level INFO or above
# is emitted.
log.basicConfig(level = "INFO")

## Define the studies

List the Recount3 studies to download, and where to cache the raw files retrieved from Recount3 (these are not needed once the read counts have been computed).

In [2]:
# Define the Recount3 studies we want to download: one TCGA cancer
# type (glioblastoma multiforme, GBM) and three SRA studies profiling
# glioblastoma patient samples (identified by searching the SRA
# metadata for "glioblastoma"/"glioma" and picking studies that survive
# curation as genuine patient-tumor cohorts).
studies = \
    [{"recount3_project_name" : "TCGA",
      "recount3_samples_category" : "GBM"},
     {"recount3_project_name" : "SRA",
      "recount3_samples_category" : "SRP027383"},
     {"recount3_project_name" : "SRA",
      "recount3_samples_category" : "SRP072494"},
     {"recount3_project_name" : "SRA",
      "recount3_samples_category" : "SRP141440"}]

# Set the directory where the raw files downloaded from Recount3 will
# be cached (they are not needed once the read counts have been
# computed and combined into a single data frame).
raw_wd = "recount3_raw"

## Download and convert the studies

For each study, download its metadata, raw gene counts, and QC metrics from Recount3, then convert the raw counts into read counts by dividing them by each sample's average mapped read length.

In [3]:
# Initialize an empty list to store, for each study, a data frame
# combining the samples' metadata and their gene expression data (as
# read counts).
dfs_studies = []

# For each study
for study in studies:

    # Get the name of the project and the samples' category.
    project_name = study["recount3_project_name"]
    samples_category = study["recount3_samples_category"]

    # Get the samples' metadata.
    df_metadata = \
        recount3.get_metadata(\
            project_name = project_name,
            samples_category = samples_category,
            save_metadata = True,
            wd = raw_wd)

    # Get the raw gene counts (as base-pair coverage sums).
    df_gene_sums = \
        recount3.get_gene_sums(\
            project_name = project_name,
            samples_category = samples_category,
            save_gene_sums = True,
            wd = raw_wd)

    # Get the quality control (QC) metrics, needed to convert the raw
    # counts into read counts.
    df_qc = \
        recount3.get_qc(\
            project_name = project_name,
            samples_category = samples_category,
            save_qc = True,
            wd = raw_wd)

    # Convert the raw counts into read counts by dividing them by
    # each sample's average mapped read length.
    df_read_counts = \
        recount3.get_read_counts(\
            df_raw_counts = df_gene_sums,
            avg_mapped_read_length = \
                df_qc["star.average_mapped_length"],
            do_round = True)

    # Combine the metadata with the read counts for this study.
    df_study = df_metadata.join(df_read_counts, how = "inner")

    # Add the combined data frame to the list.
    dfs_studies.append(df_study)

INFO:bulkdgd.recount3.util:Retrieving metadata from: http://duffel.rail.bio/recount3/human/data_sources/tcga/metadata/BM/GBM/tcga.tcga.GBM.MD.gz.


INFO:bulkdgd.recount3.util:The metadata with the sample/experiment attributes split into different columns were successfully written in 'recount3_raw/tcga_GBM_metadata.gz'.


INFO:bulkdgd.recount3.util:The RNA-seq data were successfully saved in 'recount3_raw/tcga_GBM_gene_sums.gz'.


INFO:bulkdgd.recount3.util:The QC metadata were successfully saved in 'recount3_raw/tcga_GBM_qc.gz'.


INFO:bulkdgd.recount3.util:Retrieving metadata from: http://duffel.rail.bio/recount3/human/data_sources/sra/metadata/83/SRP027383/sra.sra.SRP027383.MD.gz.


...

INFO:bulkdgd.recount3.util:Experiments' attributes were found in the metadata (see below).


INFO:bulkdgd.recount3.util:experiment attribute 'GEO_Accession' found. Unique values: 'GSM3106683', 'GSM3106684', 'GSM3106685', 'GSM3106686', 'GSM3106687', 'GSM3106688', 'GSM3106689', 'GSM3106690', 'GSM3106691', 'GSM3106692', 'GSM3106693', 'GSM3106694', 'GSM3106695', 'GSM3106696', 'GSM3106697', 'GSM3106698', 'GSM3106699', 'GSM3106700', 'GSM3106701', 'GSM3106702', 'GSM3106703', 'GSM3106704', 'GSM3106705', 'GSM3106706'.


INFO:bulkdgd.recount3.util:The metadata with the sample/experiment attributes split into different columns were successfully written in 'recount3_raw/sra_SRP141440_metadata.gz'.


INFO:bulkdgd.recount3.util:The RNA-seq data were successfully saved in 'recount3_raw/sra_SRP141440_gene_sums.gz'.


INFO:bulkdgd.recount3.util:The QC metadata were successfully saved in 'recount3_raw/sra_SRP141440_qc.gz'.


## Combine the studies

Concatenate the four studies into a single data frame, so that all downloaded samples can be preprocessed and used together.

In [4]:
# Concatenate the four studies into a single data frame. Since the
# studies do not all report the same metadata fields, missing values
# are introduced where a field is absent for a given study - this does
# not affect the gene expression columns, which are present for all
# samples in all studies.
df_samples = pd.concat(dfs_studies, axis = 0)

## Preprocess the samples

Preprocess the combined samples so that their genes match the set of genes included in the bulkdgd model.

In [5]:
# Preprocess the combined samples so that their genes match the set of
# genes included in the bulkdgd model (excluding genes not in the
# model and adding a count of 0 for genes in the model but missing
# from the downloaded data).
df_preproc, genes_excluded, genes_missing = \
    ioutil.preprocess_samples(df_samples = df_samples)

INFO:bulkdgd.ioutil.samplesio:64837 column(s) containing gene expression data was (were) found in the input data frame.


INFO:bulkdgd.ioutil.samplesio:893 column(s) containing additional information (not gene expression data) was (were) in the input data frame: recount3_project_name, recount3_samples_category, rail_id, study, tcga_barcode, gdc_file_id, gdc_cases.project.name, gdc_cases.project.released, gdc_cases.project.state, gdc_cases.project.primary_site, gdc_cases.project.project_id, cgc_sample_sample_type, gdc_cases.case_id, gdc_updated_datetime, gdc_file_name, gdc_submitter_id, gdc_file_size, gdc_access, gdc_platform, gdc_state, gdc_type, gdc_file_state, gdc_experimental_strategy, gdc_data_type, gdc_md5sum, gdc_data_format, gdc_data_category, gdc_center.code, gdc_center.name, gdc_center.short_name, gdc_center.center_id, gdc_center.namespace, gdc_center.center_type, gdc_metadata_files.updated_datetime.analysis, gdc_metadata_files.created_datetime.analysis, gdc_metadata_files.file_name.analysis, gdc_metadata_files.md5sum.analysis, gdc_metadata_files.data_format.analysis, gdc_metadata_files.access.an

INFO:bulkdgd.ioutil.samplesio:Now looking for duplicated samples...


INFO:bulkdgd.ioutil.samplesio:No duplicated samples were found.


INFO:bulkdgd.ioutil.samplesio:Now looking for missing values in the columns containing gene expression data...


...

INFO:bulkdgd.ioutil.samplesio:Now looking for duplicated genes...


INFO:bulkdgd.ioutil.samplesio:No duplicated genes were found.


INFO:bulkdgd.ioutil.samplesio:In the data frame containing the pre-processed samples, the columns containing gene expression data will be ordered according to the list of genes included in the DGD model (taken from '/home/lcb518/software/bulkdgd-2.0.0/bulkdgd/data/model/genes/genes.txt').


INFO:bulkdgd.ioutil.samplesio:In the data frame containing the pre-processed samples, the columns found in the input data frame which did not contain gene expression data, if any were present, will be appended as the last columns of the data frame and appear in the same order as they did in the input data frame.


INFO:bulkdgd.ioutil.samplesio:All genes included in the DGD model were found in the input samples.


## Save the outputs

Save the full set of preprocessed samples (not tracked in the repository because of its size), plus a small, illustrative subset (five samples per study) that is checked in and can be used directly in :doc:`Tutorial 1 <tutorial_1>`.

In [6]:
# Save the full set of preprocessed samples (all downloaded samples
# from all four studies) - this file is not tracked in the repository
# because of its size, but it is what a real analysis would use.
ioutil.save_samples(\
    df = df_preproc,
    csv_file = f"{raw_wd}/samples_preprocessed_full.csv",
    sep = ",")

# For the tutorial, save only a small, illustrative subset of the
# preprocessed samples (the first five samples of each study), so that
# the file checked into the repository stays small while still
# containing samples from all four studies.
df_preproc_subset = \
    df_preproc.groupby(\
        ["recount3_project_name", "recount3_samples_category"],
        group_keys = False).head(5)

ioutil.save_samples(\
    df = df_preproc_subset,
    csv_file = "samples_preprocessed.csv",
    sep = ",")

# Save the lists of genes excluded from/missing in the downloaded
# data (these are the same regardless of how many samples are kept).
with open("genes_excluded.txt", "w") as f:
    f.write("\n".join(genes_excluded))

with open("genes_missing.txt", "w") as f:
    f.write("\n".join(genes_missing))

## Summary

Report how many samples were downloaded for each study, and how the preprocessing step changed the number of genes.

In [7]:
# Report how many samples were downloaded for each study, and how the
# preprocessing step changed the number of genes.
log.info(\
    "Samples downloaded per study:\n" + \
    df_samples.groupby(\
        ["recount3_project_name",
         "recount3_samples_category"]).size().to_string())

log.info(f"Number of genes excluded: {len(genes_excluded)}.")
log.info(f"Number of genes missing (added with a count of 0): " \
         f"{len(genes_missing)}.")
log.info(f"Number of samples in the tutorial's subset: " \
         f"{len(df_preproc_subset)}.")

INFO:root:Samples downloaded per study:
recount3_project_name  recount3_samples_category
SRA                    SRP027383                    274
                       SRP072494                     36
                       SRP141440                     64
TCGA                   GBM                          175


INFO:root:Number of genes excluded: 49942.


INFO:root:Number of genes missing (added with a count of 0): 0.


INFO:root:Number of samples in the tutorial's subset: 20.
